# 09. Проверяем присланный скрипт по времени

В скрипте есть хорошие идеи: самая частая платформа, тип продавца, день недели и свойства запросов. Там же есть разброс координат, но нули считаются как доля событий, а пропуски стандартного отклонения превращаются в ноль. Сначала воспроизведём его разбиение, затем оценим те же группы на нескольких следующих друг за другом отрезках. Во вставленном коде повреждены строки с `train_features` и проверка `__name__`, поэтому воспроизводим сам подход, а не эти опечатки

In [1]:
# Открой ноутбук из корня проекта или из папки notebooks.
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "data" / "train.csv").is_file():
    raise FileNotFoundError("Не найдена папка data. Открой ноутбук из проекта.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from utils.features import build_features
from utils.journeys import build_journey_features
from utils.pasted_features import build_pasted_features
from utils.metric import precision_at_recall
from utils.time_validation import FOLDS, walk_forward_splits

RANDOM_STATE = 42

## Готовим признаки

В каждом случае читаем события только до конца окна куки. Самый частый город, платформа и тип продавца становятся строковыми категориями. В запросах смотрим среднюю длину и долю запросов с цифрами. День недели берём от первого события. Это доступно при окончании суточного окна, как требует условие

In [2]:
parse_dates = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv("data/train.csv", parse_dates=parse_dates)
test = pd.read_csv("data/test.csv", parse_dates=parse_dates)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])
base_features, groups, n_inside = build_features(train, test, events)
journey_features, journey_groups = build_journey_features(train, test, events)
extra = build_pasted_features(train, test, events)
features = base_features.merge(
    journey_features.drop(columns="split"), on="cookie_id", validate="one_to_one"
).merge(extra, on="cookie_id", validate="one_to_one")
fit = features.iloc[:len(train)].reset_index(drop=True).merge(
    train[["cookie_id", "window_end_ts"]], on="cookie_id", validate="one_to_one"
)
y = train.target.to_numpy()
assert fit.cookie_id.equals(train.cookie_id)

base = [column for column in groups["base"]
        if column not in {"headless_ua", "automation_ua", "ua_count"}]
routes = base + groups["routes"]
context = routes + [column for column in journey_groups["context"]
                    if column != "seller_switch_share"]
start_columns = context + ["top_location_cat"]
feature_groups = {
    "platform_seller": ["top_platform_cat", "top_seller_cat"],
    "calendar": ["is_weekend", "day_of_week_cat"],
    "pointer": ["pointer_x_std_script", "pointer_y_std_script", "pointer_x_zeros"],
    "query": ["query_mean_len", "query_has_digits"],
}
cat_features = ["top_location_cat", "top_platform_cat", "top_seller_cat", "day_of_week_cat"]

def make_model(columns):
    return CatBoostClassifier(
        iterations=600, depth=6, learning_rate=.05, l2_leaf_reg=5,
        loss_function="Logloss", verbose=False, random_seed=RANDOM_STATE,
        thread_count=4, allow_writing_files=False,
        cat_features=[column for column in cat_features if column in columns],
    )

print("Кук:", len(features), "| событий в окне:", n_inside)
print("Базовых признаков скрипта:", len(start_columns))

Кук: 16000 | событий в окне: 288126
Базовых признаков скрипта: 62


## Что показывает разбиение из скрипта

Скрипт сортирует куки по началу окна и берёт последние 20% строк. Это разрезает 16 апреля пополам: часть кук за день попадает в обучение, часть в проверку. Ниже считаем результат такого подхода для понимания числа 0,75, но для выбора модели его использовать нельзя. Скрипт ещё и выбирает группы признаков по этой же проверочной части, что делает результат оптимистичнее

In [3]:
sorted_fit = fit.assign(target=y).sort_values("window_start_ts").reset_index(drop=True)
val_size = int(len(sorted_fit) * .2)
train_part = sorted_fit.iloc[:-val_size]
valid_part = sorted_fit.iloc[-val_size:]
print("Обучение:", len(train_part), "| проверка:", len(valid_part))
print("Последний день в обучении:", train_part.window_start_ts.max())
print("Первый день в проверке:", valid_part.window_start_ts.min())
print("Кук 16 апреля в обучении:", train_part.window_start_ts.eq("2026-04-16").sum())
print("Кук 16 апреля в проверке:", valid_part.window_start_ts.eq("2026-04-16").sum())

split_rows = []
for name, columns in {"base": start_columns, **{
    group: start_columns + added for group, added in feature_groups.items()
}}.items():
    model = make_model(columns)
    model.fit(train_part[columns], train_part.target)
    score = model.predict_proba(valid_part[columns])[:, 1]
    split_rows.append({"variant": name,
                       "p_at_r70": precision_at_recall(valid_part.target, score)})
split_table = pd.DataFrame(split_rows).set_index("variant")
display(split_table.round(3))

base_score = split_table.loc["base", "p_at_r70"]
kept = [name for name in feature_groups if split_table.loc[name, "p_at_r70"] > base_score]
selected_columns = start_columns + [column for name in kept for column in feature_groups[name]]
selected_model = make_model(selected_columns)
selected_model.fit(train_part[selected_columns], train_part.target)
selected_score = selected_model.predict_proba(valid_part[selected_columns])[:, 1]
print("Группы, выбранные по тому же holdout:", kept)
print("Оценка после выбора:", round(precision_at_recall(valid_part.target, selected_score), 3))

Обучение: 8873 | проверка: 2218
Последний день в обучении: 2026-04-16 00:00:00
Первый день в проверке: 2026-04-16 00:00:00
Кук 16 апреля в обучении: 473
Кук 16 апреля в проверке: 267


/var/folders/h3/56vx48wx5wvgfvplsq8hvkf00000gn/T/ipykernel_46193/3909245619.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sorted_fit = fit.assign(target=y).sort_values("window_start_ts").reset_index(drop=True)


,p_at_r70
variant,
base,0.487
platform_seller,0.518
calendar,0.536
pointer,0.762
query,0.498


Группы, выбранные по тому же holdout: ['platform_seller', 'calendar', 'pointer', 'query']
Оценка после выбора: 0.727


## Вместо одного разреза: проверка на будущих днях

Выбираем walk-forward CV. В каждом из трёх раундов модель видит только куки, чьи суточные окна закончились к началу проверки. Новые дни остаются целиком в одной части. Обычный bootstrap здесь не подходит для оценки будущего: случайно смешанные куки из разных дат дадут модели сведения из будущего. Блочный bootstrap мог бы оценивать разброс уже полученной метрики, но не заменит такую проверку.

Для ясного сравнения фиксируем варианты заранее: исходный набор скрипта, он плюс координаты, координаты с каждой дополнительной группой отдельно и все группы вместе. После обучения на первых двух проверочных отрезках можно выбрать вариант, а третий оставить последней проверкой выбора

In [4]:
configs = {
    "script_base": start_columns,
    "script_pointer": start_columns + feature_groups["pointer"],
    "pointer_platform": start_columns + feature_groups["pointer"] + feature_groups["platform_seller"],
    "pointer_calendar": start_columns + feature_groups["pointer"] + feature_groups["calendar"],
    "pointer_query": start_columns + feature_groups["pointer"] + feature_groups["query"],
    "script_all": start_columns + [column for group in feature_groups.values() for column in group],
}
rows = []
predictions = []
for name, columns in configs.items():
    for fold_name, fit_mask, valid_mask in walk_forward_splits(fit):
        model = make_model(columns)
        model.fit(fit.loc[fit_mask, columns], y[fit_mask])
        score = model.predict_proba(fit.loc[valid_mask, columns])[:, 1]
        rows.append({"variant": name, "fold": fold_name,
                     "p_at_r70": precision_at_recall(y[valid_mask], score)})
        predictions.append(pd.DataFrame({
            "cookie_id": fit.loc[valid_mask, "cookie_id"].to_numpy(),
            "fold": fold_name, "target": y[valid_mask],
            "variant": name, "score": score,
        }))
    print("Проверено:", name)

results = pd.DataFrame(rows)
cv_table = results.pivot(index="variant", columns="fold", values="p_at_r70")
cv_table["среднее"] = cv_table.mean(axis=1)
display(cv_table[[name for name, _, _ in FOLDS] + ["среднее"]].round(3))

Проверено: script_base


Проверено: script_pointer


Проверено: pointer_platform


Проверено: pointer_calendar


Проверено: pointer_query


Проверено: script_all


fold,11–13 апреля,14–16 апреля,17–19 апреля,среднее
variant,,,,
pointer_calendar,0.639,0.714,0.709,0.687
pointer_platform,0.636,0.714,0.709,0.686
pointer_query,0.576,0.708,0.737,0.673
script_all,0.624,0.706,0.724,0.685
script_base,0.515,0.544,0.533,0.531
script_pointer,0.631,0.695,0.720,0.682


In [5]:
first_two = [name for name, _, _ in FOLDS[:2]]
last = FOLDS[-1][0]
chosen = cv_table[first_two].mean(axis=1).idxmax()
print("Выбор по первым двум проверкам:", chosen)
print("Среднее на них:", round(cv_table.loc[chosen, first_two].mean(), 3))
print("Проверка на последних днях:", round(cv_table.loc[chosen, last], 3))

keys = ["cookie_id", "fold", "target"]
wide = pd.concat(predictions).pivot(index=keys, columns="variant", values="score").reset_index()
prior = pd.read_csv("output/pointer_location_validation_scores.csv")
wide = wide.merge(prior[keys + ["context_plus", "base_plus", "routes_plus"]],
                  on=keys, validate="one_to_one")
wide["previous_rank_ensemble"] = wide.groupby("fold")[[
    "base_plus", "routes_plus", "context_plus"
]].rank(pct=True).mean(axis=1)
benchmark_rows = []
for fold_name, part in wide.groupby("fold"):
    benchmark_rows.append({
        "fold": fold_name,
        "одна старая + курсор и город": precision_at_recall(part.target, part.context_plus),
        "ансамбль с курсором и городом": precision_at_recall(part.target, part.previous_rank_ensemble),
    })
benchmark = pd.DataFrame(benchmark_rows).set_index("fold").reindex([f[0] for f in FOLDS])
benchmark.loc["среднее"] = benchmark.mean()
display(benchmark.round(3))

output = Path("output/pasted_script_cv_scores.csv")
wide.to_csv(output, index=False, float_format="%.17g")
print("Сохранены оценки:", output)

Выбор по первым двум проверкам: pointer_calendar
Среднее на них: 0.676
Проверка на последних днях: 0.709


,одна старая + курсор и город,ансамбль с курсором и городом
fold,,
11–13 апреля,0.657,0.680
14–16 апреля,0.714,0.688
17–19 апреля,0.778,0.806
среднее,0.716,0.725


Сохранены оценки: output/pasted_script_cv_scores.csv


## Вывод

На разбиении из присланного скрипта группа координат дала **0,762**. Это воспроизводит число около 0,75. Но 16 апреля попало и в обучение, и в проверку. К тому же скрипт выбирает группы признаков по этому же куску: все четыре по отдельности оказались лучше его базы, а вместе дали уже **0,727** на той же проверке. Такой результат нельзя считать независимой оценкой.

На трёх проверках по целым будущим дням вариант скрипта с курсором получил **0,631 / 0,695 / 0,720**, среднее **0,682**. Добавление всех его новых групп дало **0,624 / 0,706 / 0,724**, среднее **0,685**. Лучший из проверенных здесь вариантов с календарём дал **0,687** в среднем; по первым двум отрезкам он набрал **0,676**, а на оставленном последнем **0,709**. Наш ансамбль с курсором и городом получил **0,680 / 0,688 / 0,806**, среднее **0,725**.

Идеи из скрипта полезны, особенно координаты, но цифра с одного разреза завышает уверенность. Для выбора модели оставляем walk-forward кросс-валидацию. Затем, когда вариант выбран, обучаем его на всём размеченном train и считаем отдельный test. Bootstrap со случайным перемешиванием кук для этого не подходит: он переносит информацию о будущих днях в обучение. Качество на скрытом test пока неизвестно